In [137]:
import pandas as pd
import numpy as np
import ipywidgets as widgets

In [138]:
books=pd.read_csv('https://storage.googleapis.com/book-recommender-system/Books.csv',on_bad_lines='skip')
users=pd.read_csv('https://storage.googleapis.com/book-recommender-system/Users.csv',on_bad_lines='skip')
ratings=pd.read_csv('https://storage.googleapis.com/book-recommender-system/Ratings.csv',on_bad_lines='skip')

/tmp/ipykernel_18097/891755940.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books=pd.read_csv('https://storage.googleapis.com/book-recommender-system/Books.csv',on_bad_lines='skip')


In [139]:
print('books',books.shape,'\n','users',users.shape,'\n','ratings',ratings.shape)

books (271360, 8) 
 users (278858, 3) 
 ratings (1149780, 3)


In [140]:
books=books[['ISBN','Book-Title','Book-Author','Year-Of-Publication','Publisher']]

In [141]:
books.rename(columns={'Book-Title':'title','Book-Author':'author','Year_Of-Publication':'year','Publisher':'publisher'},inplace=True)

In [142]:
users=users.rename(columns={'User-ID':'user_id','Location':'location','Age':'age'})

In [143]:
ratings.rename(columns={'User-ID':'user_id','Book-Rating':'rating'},inplace=True)

# **Filter Users**

In [144]:
valid_users=ratings['user_id'].value_counts()>200
valid_users.shape

(105283,)

In [145]:
valid_ids=valid_users[valid_users].index



In [146]:
fratings=ratings[ratings['user_id'].isin(valid_ids)]
fratings.shape

(526356, 3)

In [147]:
ratings_with_books=fratings.merge(books,on='ISBN')
ratings_with_books.shape

(487671, 7)

In [148]:
number_rating=ratings_with_books.groupby('title')['rating'].count().reset_index().rename(columns={'rating':'no_of_ratings'})
number_rating.shape

(160269, 2)

In [149]:
final_rating=ratings_with_books.merge(number_rating,on='title')
final_rating.shape

(487671, 8)

# **Filter Books**

In [150]:
final_rating=final_rating[final_rating['no_of_ratings']>=50]

In [151]:
final_rating.drop_duplicates(['user_id','title'],inplace=True)
final_rating.shape

(59850, 8)

# **MAKE PIVOT TABLE**

In [152]:
book_pivot=final_rating.pivot_table(columns='user_id',index='title',values='rating')

In [153]:
book_pivot.fillna(0,inplace=True)
book_pivot.shape

(742, 888)

# **MAKING ML MODEL**

In [154]:
from scipy.sparse import csr_matrix

In [155]:
book_sparse=csr_matrix(book_pivot)

In [156]:
from sklearn.neighbors import NearestNeighbors
model=NearestNeighbors(algorithm='brute')

In [157]:
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [158]:
def recommender(book_name):
  matching_indices = np.where(book_pivot.index == book_name)[0]
  if len(matching_indices) == 0:
    print(f"Book '{book_name}' not found in the database. Please check the title and try again.")
    return
  book_id = matching_indices[0]
  distances,suggestions=model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1),n_neighbors=6)
  print(f"The suggestions for {book_name} are: ")
  for i in range(len(suggestions[0])):
    if i != 0:
      print(book_pivot.index[suggestions[0][i]])
  return

# ***MAKING A VISUAL RECOMMENDER SYSTEM***

In [159]:
from IPython.display import display
book_input = widgets.Text(value='',placeholder='Type a book name here...',description='Book Title:',disabled=False)
submit_button = widgets.Button(description="Get Recommendations",button_style='success')

In [160]:
def on_button_clicked(b):
  book_name_input=book_input.value
  if book_name_input.strip()=='':
    print('Please enter a valid book title')
    return
  print(f"Searching recommendations for {book_name_input}...\n")
  try:
    recommender(book_name_input)
  except Exception as e:
    print(f"Book not found or error occured:{e}")
submit_button.on_click(on_button_clicked)
display(book_input, submit_button)


Text(value='', description='Book Title:', placeholder='Type a book name here...')

Button(button_style='success', description='Get Recommendations', style=ButtonStyle())

Searching recommendations for Exclusive...

The suggestions for Exclusive are: 
The Cradle Will Fall
Jacob Have I Loved
Fine Things
The Long Road Home
No Safe Place


# ***Transformation Steps***


In [161]:
books.columns

Index(['ISBN', 'title', 'author', 'Year-Of-Publication', 'publisher'], dtype='object')

In [162]:
books.head()

,ISBN,title,author,Year-Of-Publication,publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


In [163]:
users.columns

Index(['user_id', 'location', 'age'], dtype='object')

In [10]:
users.head()

,user_id,location,age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [12]:
ratings.columns

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')

In [14]:
ratings.head()

,user_id,ISBN,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [15]:
print('books',books.shape,'\n','users',users.shape,'\n','ratings',ratings.shape)

books (271360, 5) 
 users (278858, 3) 
 ratings (1149780, 3)


In [27]:
ratings['user_id'].value_counts().shape

(105283,)

In [34]:
valid_users[valid_users].shape

(899,)

In [164]:
valid_ids

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
       116122,  44296,  28634,  59727,  73681, 274808, 188951,   9856, 155916,
       268622],
      dtype='int64', name='user_id', length=899)

In [40]:
fratings.shape

(526356, 3)

In [41]:
fratings

,user_id,ISBN,rating
1456,277427,002542730X,10
1457,277427,0026217457,0
1458,277427,003008685X,8
1459,277427,0030615321,0
1460,277427,0060002050,0
...,...,...,...
1147612,275970,3829021860,0
1147613,275970,4770019572,0
1147614,275970,896086097,0
1147615,275970,9626340762,8


In [44]:
ratings_with_books.head()

,user_id,ISBN,rating,title,author,Year-Of-Publication,publisher
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books


In [51]:
number_rating.head()

,title,no_of_ratings
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [52]:
number_rating.shape

(160269, 2)

In [59]:
final_rating.head()

,user_id,ISBN,rating,title,author,Year-Of-Publication,publisher,no_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,13


In [64]:
final_rating.shape

(61853, 8)

In [65]:
final_rating.head()

,user_id,ISBN,rating,title,author,Year-Of-Publication,publisher,no_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,82
13,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,133
15,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,108
18,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,108
24,277427,006440188X,0,The Secret Garden,Frances Hodgson Burnett,1998,HarperTrophy,79


In [68]:
final_rating.shape

(59850, 8)

In [73]:
book_pivot.shape

(742, 888)

In [74]:
book_pivot

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [111]:
distances,suggestions=model.kneighbors(book_pivot.iloc[240,:].values.reshape(1,-1),n_neighbors=6)

In [112]:
distances

array([[ 0.        , 62.08059278, 68.05145112, 71.49125821, 72.0624729 ,
        74.16198487]])

In [113]:
suggestions

array([[240, 238, 237, 241, 239, 688]])

In [114]:
print(book_pivot.index[240])

Harry Potter and the Prisoner of Azkaban (Book 3)


In [115]:
for i in range(len(suggestions)):
  print(book_pivot.index[suggestions[i]])

Index(['Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Sorcerer's Stone (Book 1)',
       'Harry Potter and the Order of the Phoenix (Book 5)', 'Tough Cookie'],
      dtype='object', name='title')


In [116]:
#given the name of a book -> to find id
print(np.where(book_pivot.index=='Harry Potter and the Prisoner of Azkaban (Book 3)')[0][0])

240


In [131]:
recommender('Tough Cookie')

The suggestions for Tough Cookie are: 
No Safe Place
Long After Midnight
Exclusive
Lake Wobegon days
Pleading Guilty
